In [1]:
from datetime import datetime

import openmeteo_requests

class IncreaseSpeed():
  '''
  Iterator for increasing the speed with the default step of 10 km/h
  You can implement this one after Iterators FP topic

  Constructor params:
    current_speed: a value to start with, km/h
    max_speed: a maximum possible value, km/h
    step: a parameter to vary the value to increase by

  Make sure your iterator is not exceeding the maximum allowed value
  '''

  def __init__(self, current_speed: int, max_speed: int, step = 10):
    self.current_speed = current_speed
    self.max_speed = max_speed
    self.step = abs(step) if step != 0 else 10
  def __iter__(self):
    return self
  def __next__(self, step = 10):
    if self.current_speed >= self.max_speed:
      raise StopIteration
    self.current_speed = min(self.current_speed + self.step, self.max_speed)
    return self.current_speed

class DecreaseSpeed():
  '''
  Iterator for decreasing the speed with the default step of 10 km/h
  You can implement this one after Iterators FP topic

  Constructor params:
    current_speed: a value to start with, km/h

  Make sure your iterator is not going below zero
  '''

  def __init__(self, current_speed: int, min_speed: int, step=10):
    self.current_speed = current_speed
    self.min_speed = min_speed
    self.step = abs(step) if step != 0 else 10

  def __iter__(self):
    return self

  def __next__(self):
    if self.current_speed <= self.min_speed:
      raise StopIteration
    self.current_speed = max(self.current_speed - self.step, self.min_speed)
    return self.current_speed

class Car():
  '''
  Car class.
  Has a class variable for counting total amount of cars on the road (increased by 1 upon instance initialization).

  Constructor params:
    max_speed: a maximum possible speed, km/h
    current_speed: current speed, km/h (0 by default)
    state: reflects if the Car is in the parking or on the road

  Methods:
    accelerate: increases the speed using IncreaseSpeed() iterator either once or gradually to the upper_border
    brake: decreases the speed using DecreaseSpeed() iterator either once or gradually to the lower_border
    parking: if the Car is not already in the parking, removes the Car from the road
    total_cars: show the total amount of cars on the road
    show_weather: shows the current weather conditions
  '''

  cars_on_road = 0

  def __init__(self, max_speed: int, current_speed=0):
    self.max_speed = max(max_speed, 0)
    self.current_speed = min(max(current_speed, 0), self.max_speed)
    self.state = 'on the road' if self.current_speed > 0 else 'parking'
    if self.state == 'on the road':
      Car.cars_on_road += 1


  def accelerate(self, upper_border=None, step=10):
    # check for state
    # create an instance of IncreaseSpeed iterator
    # check if smth passed to upper_border and if it is valid speed value
    # if True, increase the speed gradually iterating over your increaser until upper_border is met
    # print a message at each speed increase
    # else increase the speed once
    # return the message with current speed
    initial_speed = self.current_speed
    target_speed = None
    if upper_border is not None and self.current_speed <= upper_border <= self.max_speed:
      target_speed = upper_border
    increaser = IncreaseSpeed(self.current_speed, self.max_speed if target_speed is None else target_speed, step)
    if target_speed is not None:
      for next_speed in increaser:
        print(f'INFO: Speed increases by {next_speed - self.current_speed}')
        self.current_speed = next_speed
        if self.state == 'parking' and self.current_speed > 0:
          self.state = 'on the road'
          Car.cars_on_road += 1
    else:
      try:
        next_speed = next(increaser)
        print(f'INFO: Speed increases by {next_speed - self.current_speed}')
        self.current_speed = next_speed
        if self.state == 'parking' and self.current_speed > 0:
          self.state = 'on the road'
          Car.cars_on_road += 1
      except StopIteration:
        pass
    message = f'INFO: The speed of this car has been increased from {initial_speed} to {self.current_speed}'
    print(message)
    return message


  def brake(self, lower_border=None, step=10):
    # create an instance of DecreaseSpeed iterator
    # check if smth passed to lower_border and if it is valid speed value
    # if True, decrease the speed gradually iterating over your decreaser until lower_border is met
    # print a message at each speed decrease
    # else increase the speed once
    # return the message with current speed
    initial_speed = self.current_speed
    target_speed = None
    if lower_border is not None and 0 <= lower_border <= self.current_speed:
      target_speed = lower_border
    decreaser = DecreaseSpeed(self.current_speed, 0 if target_speed is None else target_speed, step)
    if target_speed is not None:
      for next_speed in decreaser:
        print(f'INFO: Speed decreases by {self.current_speed - next_speed}')
        self.current_speed = next_speed
    else:
      try:
        next_speed = next(decreaser)
        print(f'INFO: Speed decreases by {self.current_speed - next_speed}')
        self.current_speed = next_speed
      except StopIteration:
        pass
    message = f'INFO: The speed of this car has been decreased from {initial_speed} to {self.current_speed}'
    print(message)
    return message

  # the next three functions you have to define yourself
  # one of the is class method, one - static and one - regular method (not necessarily in this order, it's for you to think)

  def parking(self):
    # gets car off the road (use state and class variable)
    # check: should not be able to move the car off the road if it's not there
    if self.state == 'parking':
      message = 'INFO: This car is already in the parking'
      print(message)
      return message
    self.brake(0)
    self.state = 'parking'
    Car.cars_on_road -= 1
    message = 'Parking the car...'
    print(message)
    return message

  @classmethod
  def total_cars(Car):
    # displays total amount of cars on the road
    return Car.cars_on_road

  @staticmethod
  def show_weather():
    # displays weather conditions
    openmeteo = openmeteo_requests.Client()
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
      'latitude': 59.9386,
      'longitude': 30.3141,
      'current': ['temperature_2m', 'apparent_temperature', 'rain', 'wind_speed_10m'],
      'wind_speed_unit': 'ms',
      'timezone': 'Europe/Moscow'
    }
    response = openmeteo.weather_api(url, params=params)[0]
    current = response.Current()
    current_temperature_2m = current.Variables(0).Value()
    current_apparent_temperature = current.Variables(1).Value()
    current_rain = current.Variables(2).Value()
    current_wind_speed_10m = current.Variables(3).Value()
    print(f"Current time: {datetime.fromtimestamp(current.Time()+response.UtcOffsetSeconds())} {response.TimezoneAbbreviation().decode()}")
    print(f'Current temperature: {round(current_temperature_2m, 0)} C')
    print(f'Current apparent_temperature: {round(current_apparent_temperature, 0)} C')
    print(f'Current rain: {current_rain} mm')
    print(f'Current wind_speed: {round(current_wind_speed_10m, 1)} m/s')


In [2]:
car1 = Car(100, 20) # max_speed = 100, initial speed = 5
car2 = Car(60, 30) # max_speed = 60, initial speed = 30
car3 = Car(100, 0) # a car that is off road upon creation
print(f"Total cars on road: {Car.total_cars()}")
# >> Total cars on road: 2


Total cars on road: 2


In [3]:
car1.accelerate(100)

# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: The speed of this car has been increased from 20 to 100
None

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


In [4]:

car2.accelerate(50)

# >> INFO: Speed increases by 10
# >> INFO: Speed increases by 10
# >> INFO: The speed of this car has been increased from 30 to 50

print("Speed of car 1:", car1.current_speed)

# >> Speed of car 1: 100

print("Speed of car 2:", car2.current_speed)

# >> Speed of car 2: 50


INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50
Speed of car 1: 100
Speed of car 2: 50


In [5]:

car1.brake(10)

# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: The speed of this car has been decreased from 100 to 10
None

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


In [6]:

car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: Speed decreases by 10
# >> INFO: The speed of this car has been decreased from 50 to 0
# >> Total cars on road: 2
# >> INFO: The speed of this car has been decreased from 0 to 0 # parking method ensures the speed is 0 in the end
# >> Parking the car...
# >> Total cars on road: 1

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [7]:

car3.accelerate(80)# car3 is now on the road
car3.show_weather()
print("Total cars on road:", Car.total_cars())

# >> INFO: The speed of this car has been increased from 0 to 80
# >> Current temperature: -0.0 C
# >> Current apparent_temperature: 14.0 C
# >> Current rain: 250.0 mm
# >> Current wind_speed: 0.0 m/s
# >> Total cars on road: 2

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80
Current time: 2026-03-14 17:15:00 GMT+3
Current temperature: 9.0 C
Current apparent_temperature: 5.0 C
Current rain: 0.0 mm
Current wind_speed: 5.5 m/s
Total cars on road: 2


In [8]:
car2.accelerate(10) # # car2 goes from parking on the road
print("Total cars on road:", Car.total_cars())

# >> INFO: Speed increases by 10
# >> INFO: The speed of this car has been increased from 0 to 10
# >> Total cars on road: 3

Car.show_weather()

# >> Current temperature: -0.0 C
# >> Current apparent_temperature: 14.0 C
# >> Current rain: 250.0 mm
# >> Current wind_speed: 0.0 m/s

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3
Current time: 2026-03-14 17:15:00 GMT+3
Current temperature: 9.0 C
Current apparent_temperature: 5.0 C
Current rain: 0.0 mm
Current wind_speed: 5.5 m/s
